# Data augmentation

In [14]:
import albumentations as A
import cv2
import os
import glob

In [15]:
# Define the Path to your Roboflow Data
INPUT_IMG_DIR = "labelled_images/train/images"
INPUT_LBL_DIR = "labelled_images/train/labels"
OUTPUT_DIR = "augmented_images"

os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels", exist_ok=True)

In [16]:
# Define the Augmentation Pipeline
# These are chosen specifically for an indoor hallway robot
transform = A.Compose([
    A.Perspective(p=0.5),              # Simulates seeing the sign at an angle
    A.RandomBrightnessContrast(p=0.4), # Simulates varying hallway lighting
    A.MotionBlur(blur_limit=7, p=0.3), # Simulates camera shake while moving
    A.GaussNoise(p=0.2),               # Simulates low-light sensor noise
    A.HorizontalFlip(p=0.5),           # Doubles data (if signs aren't text-sensitive)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

In [17]:
image_files = glob.glob(f"{INPUT_IMG_DIR}/*.jpg") + glob.glob(f"{INPUT_IMG_DIR}/*.jpeg")

print(f"Found {len(image_files)} images. Starting augmentation...")

Found 51 images. Starting augmentation...


In [18]:
for img_path in image_files:
    # Fix 1: Handle Roboflow dots/hashes correctly
    base_name = os.path.basename(img_path)
    file_name = os.path.splitext(base_name)[0] 
    
    image = cv2.imread(img_path)
    if image is None:
        print(f"Skipping {file_name}: Could not read image file.")
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Fix 2: Robust Label Parsing
    label_path = os.path.join(INPUT_LBL_DIR, f"{file_name}.txt")
    if not os.path.exists(label_path):
        print(f"Warning: Label not found for {file_name}. Skipping.")
        continue

    bboxes = []
    class_labels = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5: # Ensures we have [class, x, y, w, h]
                class_labels.append(int(float(parts[0])))
                # Slicing [1:5] strictly takes 4 coordinates, ignoring metadata
                bboxes.append([float(x) for x in parts[1:5]])

    if not bboxes:
        print(f"No valid bboxes in {file_name}.txt. Skipping.")
        continue

    # 4. GENERATE 100 VARIATIONS
    for i in range(100):
        try:
            transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            
            aug_file_name = f"{file_name}_aug_{i}"
            
            # Save Image
            cv2.imwrite(f"{OUTPUT_DIR}/images/{aug_file_name}.jpg", 
                        cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR))
            
            # Save Labels
            with open(f"{OUTPUT_DIR}/labels/{aug_file_name}.txt", "w") as f:
                for idx, box in enumerate(transformed['bboxes']):
                    # Ensure class label matches the box
                    f.write(f"{class_labels[idx]} {' '.join(map(str, box))}\n")
                    
        except Exception as e:
            print(f"Error augmenting {file_name} at iteration {i}: {e}")
            break

print(f"Success! Augmented images generated in '{OUTPUT_DIR}'")

Success! Augmented images generated in 'augmented_images'


# Split the data into train, validation and testing sets

In [4]:
import os
import random
import shutil

# Set your paths
dataset_path = "augmented_images"
output_path = "dataset_final"

# Split Ratios
train_ratio = 0.7
val_ratio = 0.2
# Test ratio will be the remaining 0.1

def split_dataset():
    # Create the YOLO directory structure
    for folder in ['images', 'labels']:
        for split in ['train', 'val', 'test']:
            os.makedirs(os.path.join(output_path, folder, split), exist_ok=True)

    # Get all image filenames (assumes .jpg, change if using .png)
    all_imgs = [f for f in os.listdir(os.path.join(dataset_path, "images")) if f.endswith('.jpg')]
    random.shuffle(all_imgs)

    # Calculate split indices
    train_count = int(len(all_imgs) * train_ratio)
    val_count = int(len(all_imgs) * val_ratio)

    train_files = all_imgs[:train_count]
    val_files = all_imgs[train_count:train_count + val_count]
    test_files = all_imgs[train_count + val_count:]

    def move_files(file_list, split_name):
        for filename in file_list:
            basename = os.path.splitext(filename)[0]
            
            # Move Image
            shutil.copy(os.path.join(dataset_path, "images", filename), 
                        os.path.join(output_path, "images", split_name, filename))
            
            # Move Label (matching the image name)
            label_name = f"{basename}.txt"
            if os.path.exists(os.path.join(dataset_path, "labels", label_name)):
                shutil.copy(os.path.join(dataset_path, "labels", label_name), 
                            os.path.join(output_path, "labels", split_name, label_name))

    move_files(train_files, 'train')
    move_files(val_files, 'val')
    move_files(test_files, 'test')
    
    print(f"Split Complete: {len(train_files)} Train | {len(val_files)} Val | {len(test_files)} Test")

split_dataset()

Split Complete: 3570 Train | 1020 Val | 510 Test
